# 02 · Train RoFormer variants (absolute + delta heads, previous-angle conditioning, QAT)
One configurable experiment. Variants (`cfg.variant`):
* `roformer` – current design (k3/s1 conv, 200 tokens, LayerNorm, post-norm)
* `patch` – strided patch embedding (`patch_stride=8` → 25 tokens, ~8× fewer MACs)
* `noattn` – patch embedding + FFN only (attention ablation)
* `causal` – patch + causal attention + RMSNorm + last-token pooling (streaming / KV-cache friendly)

Heads: `absolute`, `delta`, `both`. `cond_prev=True` feeds the previous angle estimate into the head.
`unroll_steps=K>1` trains with **integration in the loop** (K consecutive windows, own estimate fed back with noise).
`qat_epochs>0` fine-tunes with fake quantisation matching `ap_fixed<W,I>` (`act_fmt`, `weight_fmt`).
W&B key is read from Kaggle Secrets (`wandb_key`), never hard-coded.

In [ ]:
# --- locate emg_common.py: Kaggle utility script, attached dataset, or local folder ---
import os, sys, glob, json
for p in ['/kaggle/usr/lib/emg_common', '/kaggle/usr/lib/emg-common', '/kaggle/input/emg-common', '.', '..', '../kaggle_scripts', 'kaggle_scripts']:
    if os.path.exists(os.path.join(p, 'emg_common.py')):
        sys.path.insert(0, os.path.abspath(p)); break
else:
    raise ImportError('emg_common.py not found: add it as a Utility Script (File > Add utility script) or dataset')
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import emg_common as ec
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
WORK = os.environ.get('EMG_WORK', '/kaggle/working')           # EMG_* env vars exist only for local smoke tests
BASE = os.environ.get('EMG_BASE_PATH', ec.Config.base_path)
EPOCHS_OVERRIDE = int(os.environ['EMG_EPOCHS']) if 'EMG_EPOCHS' in os.environ else None
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| emg_common from', ec.__file__)

In [ ]:
# processed data: output of 01_preprocess_split (same session) or attached as a dataset
candidates = [f'{WORK}/db2_processed'] + sorted(glob.glob('/kaggle/input/*/db2_processed')) + sorted(glob.glob('/kaggle/input/*'))
npz_files = []
for c in candidates:
    npz_files = sorted(glob.glob(f'{c}/*/data.npz'))
    if npz_files: break
assert npz_files, 'no processed data found: run 01_preprocess_split first or attach its output'
print(len(npz_files), 'subjects:', [os.path.basename(os.path.dirname(f)) for f in npz_files])

In [ ]:
cfg = ec.Config(processed_dir=os.path.dirname(os.path.dirname(npz_files[0])), save_dir=f'{WORK}/outputs',
                variant='patch', patch_stride=8, heads='both', cond_prev=True, unroll_steps=0,
                d_model=128, d_ff=256, num_heads=8, num_layers=1, norm='layernorm', dropout=0.1,
                epochs=40, patience=8, lr=3e-4, batch_size=64, qat_epochs=5, act_fmt=(10, 3), weight_fmt=(10, 3),
                use_wandb=os.environ.get('EMG_WANDB', '1') == '1', project_name='ninapro_db2_roformer')
cfg.apply_variant()
if EPOCHS_OVERRIDE is not None: cfg.epochs = cfg.qat_epochs = EPOCHS_OVERRIDE
cfg.run_name = f'{cfg.variant}_{cfg.heads}_prev{int(cfg.cond_prev)}_K{cfg.unroll_steps}_qat{cfg.qat_epochs}'
print(cfg.run_name, '| tokens:', cfg.num_tokens, '| MACs:', ec.estimate_macs(cfg))
run = ec.init_wandb(cfg)

In [ ]:
model, info = ec.train_model(cfg, npz_files, run=run, tag=cfg.run_name)
hist = pd.DataFrame(info['history']); display(hist.tail())
hist.plot(x='epoch', y=['train_loss', 'val_loss'], logy=True, title=cfg.run_name); plt.show()

In [ ]:
# ---- window-level metrics (shuffled, independent windows) ----
device = ec.get_device()
eval_cfg = ec.Config.from_dict({**cfg.to_dict(), 'unroll_steps': 0})
_, val_loader = ec.make_loaders(eval_cfg, npz_files)
res = ec.predict_windows(model, val_loader, cfg, device); wm = ec.window_metrics(res); print('window metrics:', wm)
# ---- sequential metrics (ordered test runs, model fed its OWN estimate) ----
rest_pose, thr = ec.rest_pose_from_train(npz_files), ec.rest_energy_threshold(npz_files)
report = ec.evaluate_sequential(model, npz_files, cfg, device, rest_pose=rest_pose, rest_threshold=thr, feedback=True)
seq = pd.DataFrame({k: {m: v[m] for m in ('rmse', 'r2', 'mae')} for k, v in report.items()}).T.sort_values('rmse'); display(seq)
ec.wandb_log(run, {**{f'window/{k}': v for k, v in wm.items()}, **{f'seq/{k}_rmse': v['rmse'] for k, v in report.items()}})
json.dump({'cfg': cfg.to_dict(), 'window': wm, 'sequential': report, 'checkpoint': info['checkpoint']},
          open(f'{cfg.save_dir}/{cfg.run_name}_report.json', 'w'), indent=2)

In [ ]:
# ---- tracking plot on the longest test run ----
runs = ec.ordered_runs(npz_files[0], 'test'); r = max(runs, key=lambda x: len(x['angles']))
p = ec.predict_run(model, r, cfg, device, feedback=True)
fig, axes = plt.subplots(5, 2, figsize=(18, 16)); n = min(600, len(r['angles']))
for j, ax in enumerate(axes.ravel()):
    ax.plot(r['angles'][:n, j], 'k', lw=1.5, label='truth')
    if p['abs'] is not None: ax.plot(p['abs'][:n, j], 'r', alpha=.5, label='absolute head')
    if p['delta'] is not None: ax.plot(ec.integrate(p['delta'], p['start'])[:n, j], 'b', alpha=.5, label='integrated delta')
    if p['abs'] is not None and p['delta'] is not None:
        ax.plot(ec.kalman_fuse(p['delta'], p['abs'], p['start'], energy=p['energy'], rest_threshold=thr, rest_pose=rest_pose)[:n, j], 'g', lw=1.5, label='kalman + rest anchor')
    ax.set_title(ec.JOINT_NAMES[j]); ax.grid(alpha=.3)
axes[0, 0].legend(); plt.tight_layout(); plt.savefig(f'{cfg.save_dir}/{cfg.run_name}_tracking.png', dpi=120); plt.show()
if run is not None:
    import wandb; run.log({'tracking': wandb.Image(f'{cfg.save_dir}/{cfg.run_name}_tracking.png')}); run.finish()